# Player Statistics Data Preprocessing

This notebook performs the data preprocessing of scraped data for the model that predicts using player statistics. We merge and clean our raw data stored in `\data`.

The attributes I chose to use in building the model include:

**Points, Assists, Rebounds, Steals, Blocks, and Turnovers**, all on a per game basis. The model would have been improved by using statistics that further distinguish positions such as three point percentage (as guards and forwards are usually much better than centers in this department), turnovers or free throw percentage.  

However, due to these more advanced statistics rarely being recorded outside of professional games, I decided to only use the traditionally recorded statistics of points, assists, rebounds, steals and blocks as the average user who may have only played up to high school basketball would either have these attributes recorded or know a rough estimate of their numbers for these attributes. For example, american highschool varsity basketball only records these statistics: [See Here](https://www.maxpreps.com/basketball/stat-leaders/).

## Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

## (1) Merge Data

In [43]:
# Retrieving relevant datasets
dates = ['20250801']
date_pattern = '|'.join(dates)
pattern = rf"nba_current_player_stats_(?:{date_pattern})_batch[0-9]+\.csv"

# Get list of datasets in \data which match the pattern
matching_files = [f for f in os.listdir("data") if re.match(pattern, f)]

print(matching_files)

# Read in and merge data
api_player_stats = pd.DataFrame()
for file in matching_files:
    df = pd.read_csv(os.path.join("data", file))
    api_player_stats = pd.concat([api_player_stats, df])

print(len(api_player_stats))
api_player_stats.head()

['nba_current_player_stats_20250801_batch2.csv', 'nba_current_player_stats_20250801_batch3.csv', 'nba_current_player_stats_20250801_batch1.csv']
2687


,player,player_id,pos,PTS,AST,REB,STL,BLK,TOV,AST_TO,STOCKS,FIC,age,year
0,Sam Hauser,1630573,F,2.50,0.38,1.12,0.04,0.08,0.08,4.750000,0.12,4.04,24.0,2022
1,Sam Hauser,1630573,F,6.40,0.89,2.55,0.36,0.26,0.38,2.342105,0.62,10.08,25.0,2023
2,Sam Hauser,1630573,F,9.01,1.04,3.49,0.51,0.32,0.41,2.536585,0.83,13.96,26.0,2024
3,Jordan Hawkins,1641722,G,7.82,1.04,2.21,0.28,0.10,0.60,1.733333,0.38,10.85,22.0,2024
4,Jaxson Hayes,1629637,C,7.38,0.88,4.05,0.41,0.86,0.83,1.060241,1.27,12.75,20.0,2020


## (2) Clean

In [44]:
print(f'The number of rows in the dataset is: {api_player_stats.shape[0]}')
print(f'The number of columns/features in the dataset is: {api_player_stats.shape[1]}')
print(f'The number of duplicate entries in the dataset is: {api_player_stats.duplicated().sum()}')
print(f'The number of missing values in the dataset is: {api_player_stats.isna().sum().sum()}')

The number of rows in the dataset is: 2687
The number of columns/features in the dataset is: 14
The number of duplicate entries in the dataset is: 0
The number of missing values in the dataset is: 0


Identify entries with `inf`. This happens with the AST/TO (Assist to Turnover) ratio when Turnover's is 0, resulting in a division by 0, creating infinity values.

In [45]:
print(api_player_stats[api_player_stats == np.inf].count())

player        0
player_id     0
pos           0
PTS           0
AST           0
REB           0
STL           0
BLK           0
TOV           0
AST_TO       20
STOCKS        0
FIC           0
age           0
year          0
dtype: int64


* There exists 20 records with an `inf` value for AST_TO.

Remove the 20 records with `inf`.

In [46]:
# Select only numeric columns
api_player_stats_numeric = api_player_stats.select_dtypes(include=[np.number])

# Find rows with any infinity values in numeric columns
api_player_stats[np.isinf(api_player_stats_numeric).any(axis=1)]

,player,player_id,pos,PTS,AST,REB,STL,BLK,TOV,AST_TO,STOCKS,FIC,age,year
148,Quenton Jackson,1631245,G,0.67,0.67,1.33,0.33,0.00,0.0,inf,0.33,3.00,25.0,2024
188,DaQuan Jeffries,1629610,G,0.67,0.33,0.67,0.00,0.00,0.0,inf,0.00,1.67,24.0,2022
357,Luke Kornet,1628436,C,2.00,0.50,1.50,0.00,0.50,0.0,inf,0.50,4.50,26.0,2022
359,Luke Kornet,1628436,C,2.17,0.67,2.08,0.25,0.17,0.0,inf,0.42,5.34,26.0,2022
360,Luke Kornet,1628436,C,2.00,0.60,1.93,0.20,0.20,0.0,inf,0.40,4.93,26.0,2022
637,Sam Merrill,1630241,G,5.00,1.00,1.80,0.80,0.00,0.0,inf,0.80,8.60,27.0,2023
674,Shake Milton,1629003,G,1.83,0.67,1.00,0.17,0.00,0.0,inf,0.17,3.67,27.0,2024
319,Cole Swider,1631306,F,1.29,0.57,1.00,0.00,0.00,0.0,inf,0.00,2.86,24.0,2023
427,Oscar Tshiebwe,1631131,F,3.25,0.25,2.00,0.25,0.12,0.0,inf,0.37,5.87,24.0,2024
582,Dariq Whitehead,1641727,G,1.50,1.50,2.00,0.00,0.50,0.0,inf,0.50,5.50,19.0,2024


In [47]:
# Remove these rows
api_player_stats = api_player_stats[~np.isinf(api_player_stats_numeric).any(axis=1)]

## (3) Export

In [48]:
# Export api player stats
api_player_stats.to_csv(os.path.join("data", "api_player_stats.csv"), index=False)